In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()

while not (ROOT / "data_labeling").exists():
    ROOT = ROOT.parent

LABEL_ROOT = ROOT / "data_labeling" / "data_labels"

In [2]:
OLD_LABELS = (
    LABEL_ROOT
    / "simplified_data_labels"
    / "labels_2010_2018_binary.csv"
)

NEW_LABELS = (
    LABEL_ROOT
    / "simplified_data_labels"
    / "labels_2019_2026_july_binary.csv"
)

REPAIR_LABELS = (
    LABEL_ROOT
    / "boundary_supplements"
    / "labels_boundary_repair_binary.csv"
)

In [4]:
STAGE_OUTPUT = (
    LABEL_ROOT
    / "continual_stages_2010_2024_revised_chronological"
)

FUTURE_OUTPUT = (
    LABEL_ROOT
    / "future_evaluation_labels"
    / "revised_chronological_2024_2026"
)

In [5]:
# Load and combine all labels
old_data = pd.read_csv(OLD_LABELS)
new_data = pd.read_csv(NEW_LABELS)
repair_data = pd.read_csv(REPAIR_LABELS)

data = pd.concat(
    [old_data, new_data, repair_data],
    ignore_index=True,
)

In [6]:
data

,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0
...,...,...
128112,2018/12/31/HMI.m2018.12.31_19.00.00.jpg,0
128113,2018/12/31/HMI.m2018.12.31_20.00.00.jpg,0
128114,2018/12/31/HMI.m2018.12.31_21.00.00.jpg,0
128115,2018/12/31/HMI.m2018.12.31_22.00.00.jpg,0


In [7]:
timestamp_text = data["label"].str.extract(
    r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})"
)[0]

data["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y.%m.%d_%H.%M.%S",
)

data = (
    data
    .drop_duplicates("label", keep="last")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

In [8]:
data

,label,goes_class,timestamp
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0,2010-12-06 07:00:00
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0,2010-12-06 08:00:00
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0,2010-12-06 09:00:00
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0,2010-12-06 10:00:00
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0,2010-12-06 11:00:00
...,...,...,...
128112,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,0,2026-07-31 19:00:00
128113,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,0,2026-07-31 20:00:00
128114,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,0,2026-07-31 21:00:00
128115,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,0,2026-07-31 22:00:00


In [9]:
# Function to select a date range
def select_period(start, end):
    return data[
        data["timestamp"].between(start, end)
    ].copy()

In [10]:
# Create all datasets
datasets = {
    # Stage 1
    "Stage1_train.csv": select_period(
        "2010-12-06 07:00:00",
        "2017-01-06 03:00:00",
    ),
    "Stage1_holdout.csv": select_period(
        "2017-01-07 04:00:00",
        "2017-12-31 23:00:00",
    ),

    # Stage 2
    "Stage2_train.csv": select_period(
        "2018-01-02 00:00:00",
        "2022-09-05 23:00:00",
    ),
    "Stage2_holdout.csv": select_period(
        "2022-09-07 00:00:00",
        "2023-06-29 22:00:00",
    ),

    # Stage 3
    "Stage3_train.csv": select_period(
        "2023-06-30 23:00:00",
        "2024-05-06 05:00:00",
    ),
    "Stage3_holdout.csv": select_period(
        "2024-05-07 06:00:00",
        "2024-06-29 10:00:00",
    ),

    # Validation and test
    "validation_2024_2025.csv": select_period(
        "2024-06-30 11:00:00",
        "2025-06-30 23:00:00",
    ),
    "test_2025_2026_july.csv": select_period(
        "2025-07-02 00:00:00",
        "2026-07-31 23:00:00",
    ),
}

Display the final distributions

In [11]:
summary = []

for name, frame in datasets.items():
    fl = int(frame["goes_class"].sum())
    nf = len(frame) - fl

    summary.append({
        "dataset": name,
        "start": frame["timestamp"].min(),
        "end": frame["timestamp"].max(),
        "samples": len(frame),
        "FL": fl,
        "NF": nf,
        "FL_percent": 100 * fl / len(frame),
    })

summary = pd.DataFrame(summary)

display(summary)

,dataset,start,end,samples,FL,NF,FL_percent
0,Stage1_train.csv,2010-12-06 07:00:00,2017-01-06 03:00:00,46638,8645,37993,18.536387
1,Stage1_holdout.csv,2017-01-07 04:00:00,2017-12-31 23:00:00,8230,330,7900,4.009721
2,Stage2_train.csv,2018-01-02 00:00:00,2022-09-05 23:00:00,39760,1995,37765,5.017606
3,Stage2_holdout.csv,2022-09-07 00:00:00,2023-06-29 22:00:00,7016,2870,4146,40.906499
4,Stage3_train.csv,2023-06-30 23:00:00,2024-05-06 05:00:00,7352,3115,4237,42.369423
5,Stage3_holdout.csv,2024-05-07 06:00:00,2024-06-29 10:00:00,1273,814,459,63.943441
6,validation_2024_2025.csv,2024-06-30 11:00:00,2025-06-30 23:00:00,8493,4142,4351,48.769575
7,test_2025_2026_july.csv,2025-07-02 00:00:00,2026-07-31 23:00:00,9187,3018,6169,32.850767


Save the stage filesz


In [12]:
STAGE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

for name in [
    "Stage1_train.csv",
    "Stage1_holdout.csv",
    "Stage2_train.csv",
    "Stage2_holdout.csv",
    "Stage3_train.csv",
    "Stage3_holdout.csv",
]:
    output = datasets[name][
        ["label", "goes_class"]
    ]

    output.to_csv(
        STAGE_OUTPUT / name,
        index=False,
    )

    print("Created:", STAGE_OUTPUT / name)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2024_revised_chronological/Stage1_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2024_revised_chronological/Stage1_holdout.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2024_revised_chronological/Stage2_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2024_revised_chronological/Stage2_holdout.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2024_revised_chronological/Stage3_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Cont

Save validation and test files

In [13]:
FUTURE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

for name in [
    "validation_2024_2025.csv",
    "test_2025_2026_july.csv",
]:
    output = datasets[name][
        ["label", "goes_class"]
    ]

    output.to_csv(
        FUTURE_OUTPUT / name,
        index=False,
    )

    print("Created:", FUTURE_OUTPUT / name)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/future_evaluation_labels/revised_chronological_2024_2026/validation_2024_2025.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/future_evaluation_labels/revised_chronological_2024_2026/test_2025_2026_july.csv
